In [ ]:
#mount drive
from google.colab import drive
path = '/content/drive/MyDrive/Folder/KA/DATASET.xlsx'

In [ ]:
#Import library
import os
import random
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_percentage_error, r2_score)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#DATA CLEANING
df = pd.read_excel(path)
print("\n===== HEAD DATA =====")
print(df.head())

# MISSING VALUE
missing = df.isnull().sum()
print("\n===== MISSING VALUES =====")
print(missing)

# DATA DUPLIKAT
duplicate_count = df.duplicated().sum()
print("\n===== DUPLIKAT =====")
print(duplicate_count)

# STATISTIK TARGET
print("\n===== STATISTIK DATA TARGET (Y) =====")
print(df['Y'].describe())

# Outlier Target
plt.figure(figsize=(6, 4))
sns.boxplot(x=df['Y'])
plt.title("Deteksi Outlier")
plt.show()

Q1 = df['Y'].quantile(0.25)
Q3 = df['Y'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['Y'] < lower_bound) | (df['Y'] > upper_bound)]
print("Data Outlier yang Ditemukan:")
print(outliers)


In [ ]:
# PEMISAHAN X & Y
X = df.drop(columns=['Y'])
y = df['Y']

# Reproducible
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# K-FOLD SET UP
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Save root
root_save = '/content/drive/MyDrive/Hasil Predict.xlsx'
os.makedirs(root_save, exist_ok=True)

# Initialize lists for storing results across folds
all_train_loss = []
fold_metrics = []
actual_pred_all = []

In [ ]:
# LOOP K-FOLD
for fold_no, (train_idx, test_idx) in enumerate(kf.split(X), 1):

    print(f"\nFold {fold_no}")
    # Split Data
    X_train = X.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    #Scaler X & Y
    scaler_x = StandardScaler()
    X_train_sc = scaler_x.fit_transform(X_train)
    X_test_sc = scaler_x.transform(X_test)

    scaler_y = StandardScaler()
    y_train_sc = scaler_y.fit_transform(y_train)

    # Build Model ANN
    model = Sequential()
    # Input + Hidden 1
    model.add(Input(shape=(X_train_sc.shape[1],)))
    model.add(Dense(20, activation='tanh'))
    model.add(Dense(10, activation='tanh'))
    # Output
    model.add(Dense(1, activation='linear'))

    # Compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

    # Early stopping
    early_stop = EarlyStopping(
    monitor='loss',
    patience=20,
    restore_best_weights=True)

    # Train Model
    history= model.fit(X_train_sc, y_train_sc,
                       epochs=150, batch_size=8,
                       callbacks=[early_stop], verbose=0)

    # Simpan loss
    all_train_loss.append(history.history['loss'])

    # Data Train
    y_train_pred_sc = model.predict(X_train_sc, verbose=0)
    # Balik ke skala asli
    y_train_pred = scaler_y.inverse_transform(y_train_pred_sc).flatten()
    y_train_true = y_train.values.flatten()

    # METRIK TRAIN
    train_mse = mean_squared_error(y_train_true, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_mape = mean_absolute_percentage_error(y_train_true,y_train_pred)*100
    train_r2 = r2(y_train_true,y_train_pred)

    # Data Test
    y_pred_sc = model.predict(X_test_sc, verbose=0) #masih scaled
    # balik ke skala asli
    y_pred = scaler_y.inverse_transform(y_pred_sc).flatten() # array 1D
    y_true = y_test.values.flatten()

    # METRIK TEST
    test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    test_mse = mean_squared_error(y_true, y_pred)
    test_mape = mean_absolute_percentage_error(y_true, y_pred)*100
    test_r2 = r2(y_true, y_pred)

    # simpan metric
    fold_metrics.append({
    "Fold": fold_no,
    # TRAIN
    "Train_RMSE": train_rmse,
    "Train_MSE": train_mse,
    "Train_MAPE": train_mape,
    "Train_R2": train_r2,
    # TEST
    "Test_RMSE": test_rmse,
    "Test_MSE": test_mse,
    "Test_MAPE": test_mape,
    "Test_R2": test_r2
    })

    # tampilkan metric tiap fold
    print("\n--- TRAIN ---")
    print(f"RMSE     : {train_rmse:.4f}")
    print(f"MSE      : {train_mse:.4f}")
    print(f"MAPE     : {train_mape:.4f}")
    print(f"R2       : {train_r2:.4f}")

    print("\n--- TEST ---")
    print(f"RMSE     : {test_rmse:.4f}")
    print(f"MSE      : {test_mse:.4f}")
    print(f"MAPE     : {test_mape:.4f}")
    print(f"R2       : {test_r2:.4f}")

    # Save actual & pred
    train_df = pd.DataFrame({
    "Actual": y_train_true,
    "Predicted": y_train_pred,
    "Dataset": "Train",
    "Fold": fold_no})

    # Save model
    os.makedirs(root_save, exist_ok=True)
    model.save(f"{root_save}/model_fold_{fold_no}.keras")
    # Simpan scaler
    joblib.dump(scaler_x, f"{root_save}/scaler_x_fold_{fold_no}.pkl")
    joblib.dump(scaler_y, f"{root_save}/scaler_y_fold_{fold_no}.pkl")

In [ ]:
# GRAFIK LOSS CURVE
plt.figure(figsize=(10,6))

# Plot tiap fold
for i in range(len(all_train_loss)):
    plt.plot(
        all_train_loss[i],
        alpha=0.3,
        linewidth=1,
        color='blue',
        label='Train Fold' if i == 0 else "")
# Cari poch terpanjang
max_epoch_train = max(len(x) for x in all_train_loss)
train_padded = []
for loss in all_train_loss:
    padded = np.pad(
        loss,
        (0, max_epoch_train - len(loss)),
        mode='edge')
    train_padded.append(padded)
avg_train_loss = np.mean(train_padded, axis=0)

# Plot rata-rata
plt.plot(
    avg_train_loss,
    color='darkblue',
    linewidth=3,
    label='Average Train Loss')
plt.title('Grafik Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig(f"{root_save}/loss_curve_training.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# GRAFIK ACTUAL VS PREDICTED TRAIN TEST
plt.figure(figsize=(7,7))
plt.scatter(
    actual_pred_all_df["Actual"],
    actual_pred_all_df["Predicted"],
    color='blue',
    alpha=0.6,
    edgecolors='k')

# Garis ideal
min_val = min(actual_pred_all_df["Actual"])
max_val = max(actual_pred_all_df["Actual"])
plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color='red',
    linestyle='--',
    linewidth=2,
    label='Perfect Prediction')

#Membuat axis simetris
plt.xlim(min_val - 0.5, max_val + 0.5)
plt.ylim(min_val - 0.5, max_val + 0.5)

plt.xlabel('Actual Y')
plt.ylabel('Predicted Y')
plt.title('Actual vs Predicted')
plt.legend()
plt.grid(True)
plt.savefig(f"{root_save}/actual_vs_predicted.png", dpi=300, bbox_inches='tight')
plt.show()